In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import json
# for content_based filtering
from sklearn.metrics.pairwise import cosine_similarity

# for collabaractive filtering

In [2]:
movieId_lookup = pd.read_csv('../dataset/processed/movieId_lookup.csv')
tmdb_movies_info = pd.read_csv('../dataset/processed/movies_df_clean.csv')



In [3]:
with open('../dataset/processed/userWatched_movies.json','r') as file:
    userWatched_movies = json.load(file)
userWatched_movies = pd.Series(userWatched_movies)

In [4]:
#  filter movies which have rated by users. 
tmdb_movies_info = tmdb_movies_info[tmdb_movies_info['movie_id'].isin(movieId_lookup['tmdb_movieId'])].reset_index(drop=True)

In [5]:
# save tmdb_movies_to database.
tmdb_movies_info.to_csv('../dataset/processed/tmdb_movies_info.csv',index=False)

In [6]:
with open('../artifact/svd_model.pkl','rb') as file:
    svd_model = pickle.load(file)

In [7]:
combined_embeddings = np.load('../artifact/embeddings/final_embeddings.npy')
director_embeddings = np.load('../artifact/embeddings/director_embeddings.npy')
keyword_embeddings = np.load('../artifact/embeddings/keyword_embeddings.npy')
genre_embeddings = np.load('../artifact/embeddings/genre_embeddings.npy')
overview_embeddings = np.load('../artifact/embeddings/overview_embeddings.npy')


In [43]:

def content_based_filtering(movie_idx,movie_size=11):
    '''
    apply content based filtering on given movie index and return the top similarities movies.
    input:
    movie_idx : index value of movieId_lookup table.

    return :
    top similarity movies list.
    '''
   
    # find the index of tmdb movie based on index value
    tmdb_movieId = movieId_lookup.iloc[movie_idx]['tmdb_movieId']
    
    # calculate movie similarity based on embedded text vector metrix.
    tmdb_index = tmdb_movies_info[tmdb_movies_info['movie_id'] == tmdb_movieId].index[0]
    similarities = cosine_similarity([combined_embeddings[tmdb_index]],combined_embeddings)[0]

    # sort index based on values then change the order into descending and give first 10 values index
    top_indicies = similarities.argsort()[::-1][:movie_size]

    # # calculate score of movide matching
    top_similar_movies  = []
    for  rm in top_indicies[1:]:
        
        movie_info = {'movie_title' : tmdb_movies_info.loc[rm,'title'],'index' :rm}
        
        movie_similarity = get_similarity_explaination(tmdb_index,rm)
        
        movie_info.update(movie_similarity)
        
        top_similar_movies.append(movie_info)

        

    return top_similar_movies
    
    # for similary_tmdb_id in top_indicies:
    # for movie_title in  tmdb_movies_info.loc[top_indicies,'title']:
    #     print(movie_title)
    # return tmdb_movies_info.loc[top_indicies,'title'],match_score
    
    
    return top_indicies
    

In [13]:


def get_similarity_explaination(movie_idx,remm_idx):
    gen_sim = cosine_similarity(genre_embeddings[movie_idx].reshape(1,-1),genre_embeddings[remm_idx].reshape(1,-1))[0,0]

    overview_sim = cosine_similarity(overview_embeddings[movie_idx].reshape(1,-1),overview_embeddings[remm_idx].reshape(1,-1))[0,0]

    keyword_sim = cosine_similarity(keyword_embeddings[movie_idx].reshape(1,-1),keyword_embeddings[remm_idx].reshape(1,-1))[0,0]

    final_score =(gen_sim*30 + overview_sim*40 + keyword_sim  *30 )
    

    return {
            'genre_similarity' : np.round(gen_sim, 2),
            "keyword_similarity": np.round(keyword_sim,2),
            "overview_similarity": np.round((overview_sim*100),2),      
            "final_score": np.round(final_score,2)
            }
    

In [14]:
content_based_filtering(current_movie_idx)[3]['overview_similarity']

np.float32(24.31)

In [15]:
current_movie_idx = 76
match_score = content_based_filtering(current_movie_idx) # movie index of current movie

recommended_movies_info = []
for recommended_movie in match_score:
    rm = ContentRecommendationMovie(recommended_movie['index'],recommended_movie['overview_similarity'])
    rm.calcualte_similarities(current_movie_idx)

    recommended_movies_info.append(rm)

for rmc in recommended_movies_info:
    print(f'Movie id : {rmc.movie_id}')
    print(f'Movie Title is : {rmc.movie_title}')
    print(f'Movie matching_keywords is : {rmc.matching_keywords}')
    print(f'Movie matching_gernes is : {rmc.matching_gernes}')
    print(f'Movie overview_similarity is : {rmc.overview_similarity}')
    print(f'Movie matching_director is : {rmc.matching_director}')
    print('-'*25)


Movie id : 101299
Movie Title is : The Hunger Games: Catching Fire
Movie matching_keywords is : ['based on young adult novel', 'dystopia']
Movie matching_gernes is : ['Action', 'Adventure', 'Science Fiction']
Movie overview_similarity is : 72.77999877929688
Movie matching_director is : True
-------------------------
Movie id : 70160
Movie Title is : The Hunger Games
Movie matching_keywords is : ['based on young adult novel', 'dystopia']
Movie matching_gernes is : ['Adventure', 'Science Fiction']
Movie overview_similarity is : 72.80999755859375
Movie matching_director is : False
-------------------------
Movie id : 131631
Movie Title is : The Hunger Games: Mockingjay - Part 1
Movie matching_keywords is : ['based on young adult novel', 'dystopia']
Movie matching_gernes is : ['Adventure', 'Science Fiction']
Movie overview_similarity is : 48.150001525878906
Movie matching_director is : True
-------------------------
Movie id : 80274
Movie Title is : Ender's Game
Movie matching_keywords is 

In [35]:
tmdb_movies_info[tmdb_movies_info['movie_id']==70160].index[0]

281

Index(['519', '4019', '5807', '6862', '13803', '24236', '29179', '30178',
       '37824', '40491', '43091', '44772', '46546', '47156', '48541', '49727',
       '51278', '52020', '57743', '57826', '60461', '61941', '66926', '69502',
       '71185', '75154', '79045', '87418', '88392', '91338', '96408', '102957',
       '107409', '108795', '109459', '111724', '111770', '113946', '114613',
       '119499', '120525', '129396', '130379', '144475', '147756', '148441',
       '149753', '156131', '158866', '161939'],
      dtype='str')

np.int64(49049)

In [203]:
class ContentRecommendationMovie:
    '''
     Save information and find similaraties and explaination of recommendated movie
    '''
    def __init__(self,movie_idx,overview_similarity):
        '''
            Input : Index of recommendated movie object ( tmdb table index )
        '''
        self.movie_idx = movie_idx
        self.movie_id =  tmdb_movies_info.loc[movie_idx,'movie_id']
        
        self.movie_title = None
        self.matching_keywords = None
        self.matching_gernes = None
        self.matching_director = None
        self.overview_similarity = round(overview_similarity,2)
        self.predicted_rating = None
   
    def calcualte_similarities(self,recommended_tmdb_idx):
        '''
            Calcluate the similarities between current movies information and given movies.
            and store it data into data members.
        '''

        cm = tmdb_movies_info.loc[self.movie_idx]
        rm = tmdb_movies_info.loc[recommended_tmdb_idx]
       
        
        # keywords 
        cm_keys = set(cm['keywords_clean'].strip().split(', '))
        rm_keys = set(rm['keywords_clean'].strip().split(', '))
        
        matching_keywords = list(rm_keys & cm_keys )
        matching_keywords.sort()
        
        # genres
        cm_genres = set(cm['genres_clean'].strip().split(', '))
        rm_genres = set(rm['genres_clean'].strip().split(', '))
    
        matching_gernes = list(cm_genres & rm_genres)
        matching_gernes.sort()
    
        # director matching
        cm_director = cm['crew_clean'].strip().split('Director: ')[1]
        rm_director = rm['crew_clean'].strip().split('Director: ')[1]
    
        if cm_director == rm_director:
            matching_director = True
        else:
            matching_director = False
    
        
        # Movie Overview

        
        self.movie_title = cm['title']
        self.matching_keywords = matching_keywords
        self.matching_gernes   = matching_gernes
        self.matching_director = matching_director

        return {
             'title'             : self.movie_title,
             'matching_keywords' :  self.matching_keywords,
             'matching_gernes'   : self.matching_gernes,
             'matching_director' : self.matching_director,
             'overview_similarity' : self.overview_similarity
        }
    
        

In [17]:
def calcualte_similarities(current_idx,recommended_tmdb_idx):
    
    cm = tmdb_movies_info.loc[current_idx]
    rm = tmdb_movies_info.loc[recommended_tmdb_idx]

    # keywords 
    cm_keys = set(cm['keywords_clean'].strip().split(', '))
    rm_keys = set(rm['keywords_clean'].strip().split(', '))
    
    matching_keywords = list(rm_keys & cm_keys )
    matching_keywords.sort()

    # genres
    cm_genres = set(cm['genres_clean'].strip().split(', '))
    rm_genres = set(rm['genres_clean'].strip().split(', '))

    matching_gernes = list(cm_genres & rm_genres)
    matching_gernes.sort()

    # director matching
    cm_director = cm['crew_clean'].strip().split('Director: ')[1]
    rm_director = rm['crew_clean'].strip().split('Director: ')[1]

    if cm_director == rm_director:
        matching_director = True
    else:
        matching_director = False

    # Movie Overview

    return  {
             
             'matching_keywords' :  matching_keywords,
             'matching_gernes'   : matching_gernes,
             'matching_director' : matching_director
        }
    

In [18]:
PATH_MOVIELENS = os.path.join('..','dataset','MovieLens')

movielens_movies = pd.read_csv(os.path.join(PATH_MOVIELENS,'movies.csv'))


In [19]:
def collaborative_based_filtering(user_id):
    '''
    apply collaborative based filtering on given movie index and return the top similarities movies.
    input:
    user_id : moviesLens rating user_id which we want to recommand movies.

    return :
    top similarity movies list.
    ''' 

     # find the index of movielens movie based on index value
    # get all movies which exist in dataset.
    all_movies = set(movieId_lookup['movieLens_movieId'].tolist())
    
    movies_watched = set(userWatched_movies[str(user_id)])
    
    
    if not type(movies_watched) == set:
        movie_watched = set(movie_watched)
        
    unwatched_movies = all_movies - movies_watched

    
    predictions = []
    # calculate the prediction rating for user U for unwatched movie uM
    for movie in unwatched_movies:
        pred_rating = svd_model.predict(user_id,movie).est
        predictions.append([movie,pred_rating])
       
    
    # sort the predicted rating by highly rated movie prediction
    predictions.sort(key=lambda x : x[1],reverse=True)


    
    movie_lookup = dict(zip(movieId_lookup['movieLens_movieId'],movieId_lookup['title_clean']))
    
    for movieId, rating in predictions[:11]:
        print(movie_lookup[movieId],' : ',rating)
    

    
    

In [22]:
movie_id = movieId_lookup.sample(1).index[0]
user_id = '108795'
print(f' movie_id {movie_id}')
print(f' user_id {user_id}')

 movie_id 1484
 user_id 108795


## Hybrid Filtering Model

This technique use both content based filtering and collaborative filtering for prediction accurate movies.
it take
user-id = 156131    
movie-id = 19995 (Avatar)

In [231]:
user_ids = 156131
movie_idxs  = 0

In [234]:
# calcaluate top 150 content similarity movies
top_content_matching_movies = content_based_filtering(movie_idxs,151)
movies_watcheds = set(userWatched_movies[str(user_ids)])

In [248]:
all_movies = set()
for content_movie in top_content_matching_movies:
    all_movies.add(movieId_lookup.loc[content_movie['index'],'movieLens_movieId'])

In [254]:
unWatched_movies = (all_movies - movies_watcheds)

In [257]:
predictions = []
    # calculate the prediction rating for user U for unwatched movie uM
for movie in unWatched_movies:
    pred_rating = svd_model.predict(user_id,movie).est
    predictions.append([movie,pred_rating])

In [268]:
len(predictions)

106

In [270]:
predictions.sort(key = lambda x : x[1],reverse=True)

In [274]:
top_pred = predictions[:10]
for i in top_pred:    
    i.append(movieId_lookup.loc[movieId_lookup['movieLens_movieId']== i[0],'tmdb_movieId'].index[0])
    

In [293]:
find_final_movies = []

In [294]:

for pred_item in top_pred:
    for movie in top_content_matching_movies:
        if movie['index'] == pred_item[2]:
            movie['predicted_rating'] =pred_item[1]
            find_final_movies.append(movie)
        

In [316]:
find_final_movies[0]

{'movie_title': 'Interstellar',
 'index': np.int64(72),
 'genre_similarity': np.float32(0.86),
 'keyword_similarity': np.float32(0.63),
 'overview_similarity': np.float32(31.29),
 'final_score': np.float32(57.12),
 'predicted_rating': np.float64(4.772296718781739)}

In [319]:
fan= []

In [320]:
for movie in find_final_movies:
    rm = ContentRecommendationMovie(movie['index'],movie['overview_similarity'])
    rm.calcualte_similarities(movie_idxs)
    rm.predicted_rating = movie[
    fan.append(rm)


In [325]:
fan[0].movie_id

np.int64(157336)

In [354]:
avatar = tmdb_movies_info.loc[0]

In [349]:
top_rms = get_movies_hybrid_based(19995,519)

In [361]:
print(avatar['title'])
print(f'Genres : {avatar['genres_clean']}')
print(f'keyword : {avatar['keywords_clean']}')

for obj in top_rms:
    print(obj.movie_title)
    print(obj.matching_gernes)
    print(obj.matching_keywords)
    print(round(obj.predicted_rating,2))
    print('*'*25)

Avatar
Genres :  Fantasy, Science Fiction, Action, Adventure
keyword :  space war, future, love affair, culture clash, romance, space, space travel, soldier, space colony, power relations, mind and soul, society, battle, tribe, marine, futuristic, alien, 3d, cgi, anti war, alien planet
Children of Men
['Action', 'Science Fiction']
['future']
4.08
*************************
Interstellar
['Adventure', 'Science Fiction']
['space', 'space travel']
3.96
*************************
Moon
['Science Fiction']
['future', 'space']
3.82
*************************
2001: A Space Odyssey
['Adventure', 'Science Fiction']
['space travel']
3.81
*************************
Gravity
['Science Fiction']
['space']
3.71
*************************
Her
['Science Fiction']
[]
3.69
*************************
X-Men: Days of Future Past
['Action', 'Adventure', 'Fantasy', 'Science Fiction']
[]
3.69
*************************
Predator
['Action', 'Adventure', 'Science Fiction']
['3d', 'alien']
3.63
*************************
Al

In [347]:
def get_movies_hybrid_based(tmdb_movie_id,user_id):
    # calcaluate top 150 content similarity movies
    movie_indx = movieId_lookup.loc[movieId_lookup['tmdb_movieId']==tmdb_movie_id].index[0]
    top_content_matching_movies = content_based_filtering(movie_indx,151)
    movies_watcheds = set(userWatched_movies[str(user_ids)])

    all_movies = set()
    for content_movie in top_content_matching_movies:
        all_movies.add(movieId_lookup.loc[content_movie['index'],'movieLens_movieId'])

    unWatched_movies = (all_movies - movies_watcheds)

    predictions = []
    # calculate the prediction rating for user U for unwatched movie uM
    for movie in unWatched_movies:
        pred_rating = svd_model.predict(user_id,movie).est
        predictions.append([movie,pred_rating])

    predictions.sort(key = lambda x : x[1],reverse=True)

    top_pred = predictions[:10]
    for i in top_pred:    
        i.append(movieId_lookup.loc[movieId_lookup['movieLens_movieId']== i[0],'tmdb_movieId'].index[0])

    find_final_movies = []
    
    for pred_item in top_pred:
        for movie in top_content_matching_movies:
            if movie['index'] == pred_item[2]:
                movie['predicted_rating'] =pred_item[1]
                find_final_movies.append(movie)

    fan= []
    for movie in find_final_movies:
        rm = ContentRecommendationMovie(movie['index'],movie['overview_similarity'])
        rm.calcualte_similarities(movie_idxs)
        rm.predicted_rating = movie['predicted_rating']
        fan.append(rm)

    return fan

In [343]:


avtar_tmdb_id = 19995
user_selected_id = 519

In [346]:
movie_indx = movieId_lookup.loc[movieId_lookup['tmdb_movieId']==avtar_tmdb_id].index[0]
top_content_matching_movies = content_based_filtering(movie_indx,151)
movies_watcheds = set(userWatched_movies[str(user_selected_id)])
